In [ ]:
import struct
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import cv2
import PIL.Image as Image
import torch.optim as optim
import matplotlib.colors as mcolors
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch
import torch.nn as nn

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.device_count())

#### Dataset preparation and visualization

In [ ]:
def read(filepath = "dataset"):
    with open(filepath, 'rb') as f:
        magic_num = struct.unpack('f', f.read(4))[0]
        if magic_num != 202021.25:
            raise ValueError('Magic number mismatch')
        width = struct.unpack("i", f.read(4))[0]
        height = struct.unpack("i", f.read(4))[0]
        flow = np.frombuffer(f.read(), dtype=np.float32).reshape((height, width, 2))

    return flow

##### Splitting dataset

In [ ]:
class FlyingChairsDataset(Dataset):
    def __init__(self, root='./data', split='train', split_file='./FlyingChairs_train_val.txt', resize=None):
        self.root = Path(root)
        self.resize = resize

        all_flo_files = sorted(self.root.rglob('*_flow.flo'))
        all_ids = [int(file.stem.split('_')[0]) for file in all_flo_files]

        if split_file is not None and split in ('train', 'val'):
            with open(split_file, 'r') as f:
                labels = [int(line.strip()) for line in f.readlines()]
            target = 1 if split == 'train' else 2
            self.ids = [sid for sid, lbl in zip(all_ids, labels) if lbl == target]
        else:
            self.ids = all_ids

        print(f'[{split}] {len(self.ids)} samples')

    def  __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        sid = self.ids[idx]

        img1 = np.array(Image.open(self.root / f'{sid:05d}_img1.ppm').convert('RGB'), dtype=np.float32) / 255.0
        img2 = np.array(Image.open(self.root / f'{sid:05d}_img2.ppm').convert('RGB'), dtype=np.float32) / 255.0
        flow = read(self.root / f'{sid:05d}_flow.flo')

        if self.resize is not None:
            new_w, new_h = self.resize
            H, W = img1.shape[:2]
            img1 = cv2.resize(img1, (new_w, new_h))
            img2 = cv2.resize(img2, (new_w, new_h))
            flow = cv2.resize(flow, (new_w, new_h))
            flow[..., 0] *= new_w / W
            flow[..., 1] *= new_h / H

        img1 = torch.from_numpy(img1.transpose(2, 0, 1))
        img2 = torch.from_numpy(img2.transpose(2, 0, 1))
        flow = torch.from_numpy(flow.transpose(2, 0, 1))

        return img1, img2, flow

In [ ]:
class FlyingChairsDatasetPreloaded(Dataset):
    def __init__(self, root='./data', split='train',
                 split_file='./FlyingChairs_train_val.txt', resize=None, device='cpu'):

        root = Path(root)
        all_flo_files = sorted(root.rglob('*_flow.flo'))
        all_ids = [int(file.stem.split('_')[0]) for file in all_flo_files]

        if split_file is not None and split in ('train', 'val'):
            with open(split_file, 'r') as f:
                labels = [int(line.strip()) for line in f.readlines()]
            target = 1 if split == 'train' else 2
            ids = [sid for sid, lbl in zip(all_ids, labels) if lbl == target]
        else:
            ids = all_ids

        print(f'[{split}] Preloading {len(ids)} samples into RAM...')
        self.data = []

        for i, sid in enumerate(ids):
            img1 = np.array(Image.open(root / f'{sid:05d}_img1.ppm').convert('RGB'), dtype=np.float32) / 255.0
            img2 = np.array(Image.open(root / f'{sid:05d}_img2.ppm').convert('RGB'), dtype=np.float32) / 255.0
            flow = read(root / f'{sid:05d}_flow.flo')

            if resize is not None:
                new_w, new_h = resize
                H, W = img1.shape[:2]
                img1 = cv2.resize(img1, (new_w, new_h))
                img2 = cv2.resize(img2, (new_w, new_h))
                flow = cv2.resize(flow, (new_w, new_h))
                flow[..., 0] *= new_w / W
                flow[..., 1] *= new_h / H

            img1 = torch.from_numpy(img1.transpose(2, 0, 1)).to(device)
            img2 = torch.from_numpy(img2.transpose(2, 0, 1)).to(device)
            flow = torch.from_numpy(flow.transpose(2, 0, 1)).to(device)

            self.data.append((img1, img2, flow))

            if (i + 1) % 2000 == 0:
                print(f'  {i+1}/{len(ids)} loaded...')

        print(f'[{split}] Done.')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

##### Converting to flow coordinates

Flow [H W 2]

u = flow[..., 0] -> x

v = flow[..., 1] -> y

magnitude = sqrt(u**2 + v**2)      # how far the pixel moved

angle     = arctan2(v, u)          # which direction, in radians [-pi, pi]

In [ ]:
def convert_flow_to_polar(flow):
    u = flow[0]
    v = flow[1]

    magnitude = np.sqrt(u**2 + v**2)
    angle = np.arctan2(v, u)

    return magnitude, angle

In [ ]:
# flow_np = flow_batch[0].numpy()       # (2, 384, 512)
# magnitude, angle = convert_flow_to_polar(flow_np)
#
# print(magnitude.shape)   # (384, 512)
# print(angle.shape)       # (384, 512)
# print(f'Magnitude: min={magnitude.min():.2f}, max={magnitude.max():.2f}')
# print(f'Angle:     min={angle.min():.2f},     max={angle.max():.2f}')

##### Convert flow to hsv

polar flow to RGB image using HSV encoding

Hue = direction

Saturation = magnitude

Value = fixed

In [ ]:
def convert_flow_to_hsv(angle, magnitude, value=1.0):
    hue = (angle + np.pi) / (2 * np.pi)
    saturation = magnitude / (magnitude.max() + 1e-5)

    H, W = magnitude.shape
    hsv = np.zeros((H, W, 3), dtype=np.float32)
    hsv[..., 0] = hue
    hsv[..., 1] = saturation
    hsv[..., 2] = value

    rgb = mcolors.hsv_to_rgb(hsv)

    return rgb

In [ ]:
def make_color_wheel(size=200):
    radius = size // 2
    y, x   = np.mgrid[-radius:radius, -radius:radius].astype(np.float32)

    angle     = np.arctan2(y, x)
    magnitude = np.sqrt(x**2 + y**2)
    mag_clipped = np.clip(magnitude, 0, radius)

    wheel = convert_flow_to_hsv(angle, mag_clipped, value=1.0)

    outside = magnitude > radius
    wheel[outside] = 1.0

    return wheel

In [ ]:
def visualize_flow(img1, img2, flow):
    im1 = img1.numpy().transpose(1, 2, 0)
    im2 = img2.numpy().transpose(1, 2, 0)
    fl  = flow.numpy()

    magnitude, angle = convert_flow_to_polar(fl)
    flow_rgb         = convert_flow_to_hsv(angle, magnitude)
    wheel            = make_color_wheel(size=200)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    axes[0].imshow(im1)
    axes[0].set_title('Frame 1')

    axes[1].imshow(im2)
    axes[1].set_title('Frame 2')

    axes[2].imshow(flow_rgb)
    axes[2].set_title('Optical Flow (HSV)')

    for ax in axes:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
train_set = FlyingChairsDatasetPreloaded(split='train', resize=(256, 256))
val_set   = FlyingChairsDatasetPreloaded(split='val',   resize=(256, 256))

# Data is preloaded into RAM as tensors, so num_workers=0 is correct
# (workers would just fork that RAM). pin_memory=True lets the host->GPU
# copy happen asynchronously when paired with non_blocking=True in the loop.
train_loader = DataLoader(train_set, batch_size=32, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=32, shuffle=False,
                          num_workers=0, pin_memory=True)

# check shapes
img1_batch, img2_batch, flow_batch = next(iter(train_loader))
print(img1_batch.shape)   # (32, 3, 256, 256)
print(flow_batch.shape)   # (32, 2, 256, 256)

In [ ]:
img1_sample = img1_batch[0]
img2_sample = img2_batch[0]
flow_sample = flow_batch[0]

visualize_flow(img1_sample, img2_sample, flow_sample)

#### FlowNetSimple

Input: [img1, img2] concatenated -> (6, H, W)

ENCODER:

DECODER (with predictions at each level):

##### DownBlock

DownBlock

Conv -> BN -> ReLU

Conv -> BN -> ReLU

MaxPool

In [ ]:
class DownBlock(nn.Module):
    def __init__(self, in_channel, out_channel, kernel_size):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channel, out_channel, kernel_size, padding=kernel_size // 2),
            nn.BatchNorm2d(out_channel),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channel, out_channel, kernel_size, padding=kernel_size // 2),
            nn.BatchNorm2d(out_channel),
            nn.ReLU(inplace=True),
        )
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        skip = self.conv_block(x)
        out  = self.pool(skip)
        return out, skip

##### Encoder
DownBlock(16, 7x7)  -> skip1, pool1

DownBlock(32, 5x5)  -> skip2, pool2

DownBlock(64, 3x3)  -> skip3, pool3

Conv(128) -> BN -> ReLU

Conv(128) -> BN -> ReLU  -> bottleneck

In [ ]:
class Encoder(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        # Channel sizes per assignment spec: 16 -> 32 -> 64 -> 128
        # (input is 6 channels = image1 and image2 concatenated)
        # If you have GPU headroom you can pass base=32 or 64 for the "better results" variant.
        self.db1 = DownBlock(6,        base,    kernel_size=7)
        self.db2 = DownBlock(base,     base*2,  kernel_size=5)
        self.db3 = DownBlock(base*2,   base*4,  kernel_size=3)

        self.bottleneck = nn.Sequential(
            nn.Conv2d(base*4, base*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base*8, base*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base*8),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        out1, skip1 = self.db1(x)
        out2, skip2 = self.db2(out1)
        out3, skip3 = self.db3(out2)
        bottleneck = self.bottleneck(out3)
        return bottleneck, skip1, skip2, skip3

##### Decoder

ConvTranspose(64) + concat(skip3) -> pred3 -> upsample

ConvTranspose(32) + concat(skip2) -> pred2 -> upsample

ConvTranspose(16) + concat(skip1) -> pred1 -> upsample (full resolution)

In [ ]:
class Decoder(nn.Module):
    """
    FlowNetSimple-style decoder with multi-scale predictions.
    Each level: ConvTranspose -> concat(skip + upsampled_pred_below) -> Conv1x1 -> pred
    Next level's ConvTranspose feeds off the CONCATENATED tensor.
    """
    def __init__(self, base=16):
        super().__init__()

        self.up3 = nn.ConvTranspose2d(base*8, base*4, kernel_size=2, stride=2)
        self.pred3    = nn.Conv2d(base*8, 2, kernel_size=1)
        self.up_pred3 = nn.ConvTranspose2d(2, 2, kernel_size=2, stride=2)

        self.up2 = nn.ConvTranspose2d(base*8, base*2, kernel_size=2, stride=2)
        self.pred2    = nn.Conv2d(base*4 + 2, 2, kernel_size=1)
        self.up_pred2 = nn.ConvTranspose2d(2, 2, kernel_size=2, stride=2)

        self.up1 = nn.ConvTranspose2d(base*4 + 2, base, kernel_size=2, stride=2)
        self.pred1 = nn.Conv2d(base*2 + 2, 2, kernel_size=1)

    def forward(self, bottleneck, skip1, skip2, skip3):
        x3_up = self.up3(bottleneck)
        x3    = torch.cat([x3_up, skip3], dim=1)
        p3    = self.pred3(x3)
        p3up  = self.up_pred3(p3)

        x2_up = self.up2(x3)
        x2    = torch.cat([x2_up, skip2, p3up], dim=1)
        p2    = self.pred2(x2)
        p2up  = self.up_pred2(p2)

        x1_up = self.up1(x2)
        x1    = torch.cat([x1_up, skip1, p2up], dim=1)
        p1    = self.pred1(x1)

        return p1, p2, p3

In [ ]:
class FlowNetSimple(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.encoder = Encoder(base=base)
        self.decoder = Decoder(base=base)

    def forward(self, img1, img2):
        x = torch.cat([img1, img2], dim=1)
        bottleneck, skip1, skip2, skip3 = self.encoder(x)
        p1, p2, p3 = self.decoder(bottleneck, skip1, skip2, skip3)
        return p1, p2, p3

In [ ]:
model = FlowNetSimple(base=16)

img1_dummy = torch.randn(2, 3, 256, 256)
img2_dummy = torch.randn(2, 3, 256, 256)

p1, p2, p3 = model(img1_dummy, img2_dummy)

print(p1.shape)
print(p2.shape)
print(p3.shape)

n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}')

#### Loss function — single output

Average Endpoint Error: mean Euclidean distance between predicted and GT flow vectors.

In [ ]:
def epe_single_output_loss(predicted, target):
    return torch.mean(torch.sqrt(((predicted - target) ** 2).sum(dim=1)))

In [ ]:
def train_single_output(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    for i, (image1, image2, flow) in enumerate(loader):
        image1 = image1.to(device, non_blocking=True)
        image2 = image2.to(device, non_blocking=True)
        flow   = flow.to(device,   non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            p1, p2, p3 = model(image1, image2)
            loss = epe_single_output_loss(p1, flow)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        if (i + 1) % 100 == 0:
            print(f'  [Train] Batch {i+1}/{len(loader)} | Loss: {loss.item():.4f}')

    return total_loss / len(loader)

In [ ]:
def validate(model, loader, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for i, (image1, image2, flow) in enumerate(loader):
            image1 = image1.to(device, non_blocking=True)
            image2 = image2.to(device, non_blocking=True)
            flow   = flow.to(device,   non_blocking=True)

            with torch.amp.autocast("cuda", dtype=torch.float16):
                p1, p2, p3 = model(image1, image2)
                loss = epe_single_output_loss(p1, flow)

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model     = FlowNetSimple(base=16).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scaler    = torch.amp.GradScaler("cuda")

NUM_EPOCHS = 10
train_losses = []
val_losses   = []

for epoch in range(NUM_EPOCHS):
    print(f'--- Epoch {epoch+1}/{NUM_EPOCHS} starting ---')
    train_loss = train_single_output(model, train_loader, optimizer, device)
    val_loss   = validate(model, val_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} | Train EPE: {train_loss:.4f} | Val EPE: {val_loss:.4f}')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train EPE')
plt.plot(val_losses,   label='Val EPE')
plt.xlabel('Epoch')
plt.ylabel('EPE Loss')
plt.title('Single Output - Training & Validation Loss')
plt.legend()
plt.tight_layout()
plt.savefig('loss_single_output.png', dpi=120)
plt.show()

In [ ]:
torch.save(model.state_dict(), 'flownet_single.pth')
print('Single-output model saved.')

#### Multi-scale output loss

Following the FlowNet paper, we sum the EPE loss across every prediction
resolution. Ground-truth flow is bilinearly downsampled to each prediction's
resolution and the flow magnitudes are rescaled (a 1-pixel displacement at
half resolution corresponds to 2 pixels at full resolution).

In [ ]:
def epe_multi_output_loss(predictions, target_full, weights=None):
    """
    predictions : tuple of predictions ordered high-to-low resolution
                  (e.g. (p1=full, p2=1/2, p3=1/4) from FlowNetSimple)
    target_full : full-res ground-truth flow, (B, 2, H, W)
    weights     : optional per-scale weights (default: equal)
    """
    if weights is None:
        weights = [1.0] * len(predictions)

    H_full, W_full = target_full.shape[-2:]
    total = 0.0
    for pred, w in zip(predictions, weights):
        Hp, Wp = pred.shape[-2:]
        target_s = nn.functional.interpolate(
            target_full, size=(Hp, Wp),
            mode='bilinear', align_corners=False
        ).clone()
        target_s[:, 0] *= Wp / W_full
        target_s[:, 1] *= Hp / H_full

        epe = torch.mean(torch.sqrt(((pred - target_s) ** 2).sum(dim=1)))
        total = total + w * epe
    return total

In [ ]:
def train_multi_output(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0

    for i, (image1, image2, flow) in enumerate(loader):
        image1 = image1.to(device, non_blocking=True)
        image2 = image2.to(device, non_blocking=True)
        flow   = flow.to(device,   non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            preds = model(image1, image2)
            loss  = epe_multi_output_loss(preds, flow)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        if (i + 1) % 100 == 0:
            print(f"  [Train] Batch {i+1}/{len(loader)} | Loss: {loss.item():.4f}")

    return total_loss / len(loader)

In [ ]:
# ----- Multi-scale training driver -----
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model_multi = FlowNetSimple(base=16).to(device)
optimizer   = optim.Adam(model_multi.parameters(), lr=1e-4)
scaler      = torch.amp.GradScaler("cuda")

NUM_EPOCHS = 10
train_losses_multi = []
val_losses_multi   = []

for epoch in range(NUM_EPOCHS):
    print(f'--- Epoch {epoch+1}/{NUM_EPOCHS} starting ---')
    train_loss = train_multi_output(model_multi, train_loader, optimizer, device)
    val_loss   = validate(model_multi, val_loader, device)  # full-res EPE for fair comparison

    train_losses_multi.append(train_loss)
    val_losses_multi.append(val_loss)
    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} | Train (multi-scale sum): {train_loss:.4f} '
          f'| Val (full-res EPE): {val_loss:.4f}')

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses_multi, label='Train (multi-scale sum)')
plt.plot(val_losses_multi,   label='Val (full-res EPE)')
plt.xlabel('Epoch')
plt.ylabel('EPE Loss')
plt.title('Multi-scale Output - Training & Validation Loss')
plt.legend()
plt.tight_layout()
plt.savefig('loss_multi_output.png', dpi=120)
plt.show()

In [ ]:
torch.save(model_multi.state_dict(), 'flownet_multi.pth')
print('Multi-scale model saved.')

#### Evaluation on MPI Sintel

Download Sintel (training set) from http://sintel.is.tue.mpg.de/ and extract:
```
MPI-Sintel-complete/
  training/
    clean/<scene>/frame_XXXX.png
    final/<scene>/frame_XXXX.png
    flow/<scene>/frame_XXXX.flo
```

We resize Sintel (1024x436) to 256x256 (matching our training resolution),
with flow magnitudes rescaled accordingly.

In [ ]:
class SintelDataset(Dataset):
    """MPI Sintel training-set loader. Uses the public ground-truth flow files."""
    def __init__(self, root='./MPI-Sintel-complete', pass_type='clean', resize=None):
        self.root      = Path(root)
        self.resize    = resize
        self.pass_type = pass_type

        flow_root = self.root / 'training' / 'flow'
        img_root  = self.root / 'training' / pass_type

        self.pairs = []
        for scene_dir in sorted(p for p in flow_root.iterdir() if p.is_dir()):
            scene = scene_dir.name
            for flo_file in sorted(scene_dir.glob('*.flo')):
                fnum = int(flo_file.stem.split('_')[-1])
                img1 = img_root / scene / f'frame_{fnum:04d}.png'
                img2 = img_root / scene / f'frame_{fnum+1:04d}.png'
                if img1.exists() and img2.exists():
                    self.pairs.append((img1, img2, flo_file))

        print(f'Sintel ({pass_type}): {len(self.pairs)} pairs')

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img1_path, img2_path, flo_path = self.pairs[idx]
        img1 = np.array(Image.open(img1_path).convert('RGB'), dtype=np.float32) / 255.0
        img2 = np.array(Image.open(img2_path).convert('RGB'), dtype=np.float32) / 255.0
        flow = read(flo_path)

        if self.resize is not None:
            new_w, new_h = self.resize
            H, W = img1.shape[:2]
            img1 = cv2.resize(img1, (new_w, new_h))
            img2 = cv2.resize(img2, (new_w, new_h))
            flow = cv2.resize(flow, (new_w, new_h))
            flow[..., 0] *= new_w / W
            flow[..., 1] *= new_h / H

        img1 = torch.from_numpy(img1.transpose(2, 0, 1))
        img2 = torch.from_numpy(img2.transpose(2, 0, 1))
        flow = torch.from_numpy(flow.transpose(2, 0, 1))
        return img1, img2, flow

In [ ]:
def evaluate_sintel(model, loader, device):
    """Returns (mean_epe, std_epe, all_epes_array)."""
    model.eval()
    epes = []
    with torch.no_grad():
        for image1, image2, flow in loader:
            image1 = image1.to(device, non_blocking=True)
            image2 = image2.to(device, non_blocking=True)
            flow   = flow.to(device,   non_blocking=True)

            p1, _, _ = model(image1, image2)
            epe_map     = torch.sqrt(((p1 - flow) ** 2).sum(dim=1))
            epe_per_img = epe_map.mean(dim=(1, 2))
            epes.extend(epe_per_img.cpu().tolist())

    epes = np.array(epes)
    return float(epes.mean()), float(epes.std()), epes

In [ ]:
# >>>> Adjust this path to where you extracted Sintel <<<<
SINTEL_ROOT = './MPI-Sintel-complete'

sintel_set    = SintelDataset(root=SINTEL_ROOT, pass_type='clean', resize=(256, 256))
sintel_loader = DataLoader(sintel_set, batch_size=8, shuffle=False,
                           num_workers=0, pin_memory=True)

# Use whichever model you trained (model or model_multi)
mean_epe, std_epe, all_epes = evaluate_sintel(model_multi, sintel_loader, device)
print(f'Sintel clean - FlowNetSimple : mean EPE = {mean_epe:.3f}  std = {std_epe:.3f}')

#### Farneback baseline (OpenCV)

Re-uses the same `sintel_loader` for an apples-to-apples comparison.
Tweak `winsize`, `levels`, `iterations`, `poly_n`, `poly_sigma` for Sintel.

In [ ]:
def farneback_flow(img1_uint8, img2_uint8,
                   pyr_scale=0.5, levels=3, winsize=15,
                   iterations=3, poly_n=5, poly_sigma=1.2):
    """Dense optical flow via the Gunnar Farneback algorithm."""
    g1 = cv2.cvtColor(img1_uint8, cv2.COLOR_RGB2GRAY)
    g2 = cv2.cvtColor(img2_uint8, cv2.COLOR_RGB2GRAY)
    flow = cv2.calcOpticalFlowFarneback(
        g1, g2, None,
        pyr_scale, levels, winsize, iterations, poly_n, poly_sigma, 0
    )
    return flow


def evaluate_farneback(loader, **fb_kwargs):
    epes = []
    for image1, image2, flow in loader:
        for b in range(image1.shape[0]):
            img1_np = (image1[b].numpy().transpose(1, 2, 0) * 255.0).astype(np.uint8)
            img2_np = (image2[b].numpy().transpose(1, 2, 0) * 255.0).astype(np.uint8)
            gt      = flow[b].numpy().transpose(1, 2, 0)

            pred = farneback_flow(img1_np, img2_np, **fb_kwargs)

            epe_map = np.sqrt(((pred - gt) ** 2).sum(axis=-1))
            epes.append(float(epe_map.mean()))

    epes = np.array(epes)
    return float(epes.mean()), float(epes.std()), epes

In [ ]:
fb_params = dict(pyr_scale=0.5, levels=3, winsize=15,
                 iterations=3, poly_n=5, poly_sigma=1.2)

mean_fb, std_fb, _ = evaluate_farneback(sintel_loader, **fb_params)
print(f'Sintel clean - Farneback     : mean EPE = {mean_fb:.3f}  std = {std_fb:.3f}')
print(f'  (params: {fb_params})')

#### Own video — subjective test

Put a short clip in the project folder (a few seconds, some motion). The cell
below produces a side-by-side mp4: original frame on the left, predicted
optical flow (HSV) on the right.

In [ ]:
def process_video(model, video_path, output_path, device, target_size=(256, 256)):
    """Run the model on consecutive frame pairs and write a side-by-side
    visualization (original | flow) at the source's fps."""
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f'Could not open {video_path}')
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0

    ret, prev = cap.read()
    if not ret:
        cap.release()
        raise IOError('Empty video')

    out_W, out_H = target_size
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(str(output_path), fourcc, fps, (out_W * 2, out_H))

    model.eval()
    frame_idx = 0
    with torch.no_grad():
        while True:
            ret, curr = cap.read()
            if not ret:
                break

            img1_rgb = cv2.cvtColor(cv2.resize(prev, target_size), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.
            img2_rgb = cv2.cvtColor(cv2.resize(curr, target_size), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.

            t1 = torch.from_numpy(img1_rgb.transpose(2, 0, 1)).unsqueeze(0).to(device)
            t2 = torch.from_numpy(img2_rgb.transpose(2, 0, 1)).unsqueeze(0).to(device)

            with torch.amp.autocast("cuda", dtype=torch.float16):
                p1, _, _ = model(t1, t2)

            flow_np  = p1[0].cpu().float().numpy()
            mag, ang = convert_flow_to_polar(flow_np)
            flow_rgb = convert_flow_to_hsv(ang, mag)

            flow_bgr = cv2.cvtColor((flow_rgb * 255).astype(np.uint8), cv2.COLOR_RGB2BGR)
            img_bgr  = cv2.resize(curr, target_size)
            writer.write(np.hstack([img_bgr, flow_bgr]))

            prev = curr
            frame_idx += 1
            if frame_idx % 30 == 0:
                print(f'  processed {frame_idx} frames')

    cap.release()
    writer.release()
    print(f'Saved {frame_idx} frames -> {output_path}')

In [ ]:
# >>>> Adjust paths <<<<
INPUT_VIDEO  = 'input_video.mp4'
OUTPUT_VIDEO = 'output_flow.mp4'

process_video(model_multi, INPUT_VIDEO, OUTPUT_VIDEO, device, target_size=(256, 256))